In [ ]:
import pandas as pd
import numpy as np


def load_table(path):
    path = str(path)
    if path.lower().endswith(('.csv',)):
        return pd.read_csv(path)
    elif path.lower().endswith(('.xls', '.xlsx')):
        return pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported extension for {path}. Use .csv, .xls, or .xlsx")


def describe_col_type(series):
    s = series.rename('v')
    non_null = s.dropna()
    if non_null.empty:
        return 'x_nan'  # only nan/None values

    # numeric conversion check
    numeric = pd.to_numeric(non_null, errors='coerce')
    is_numeric = numeric.notna().all()

    if is_numeric:
        # check integerness of numeric values
        if np.all(np.floor(numeric) == numeric):
            return 'x_int'
        else:
            return 'x'

    # non-numeric at all -> string category/label
    return 'x_str'


def make_descriptor_encoding(desc_df, key='SMILES'):
    enc = {}
    x_i = 0
    x_str_j = 0
    x_nan_k = 0
    x_int_m = 0

    for col in desc_df.columns:
        if col == key:
            continue
        col_type = describe_col_type(desc_df[col])

        if col_type == 'x':
            enc[col] = f'x_{x_i}'
            x_i += 1
        elif col_type == 'x_str':
            enc[col] = f'x_str_{x_str_j}'
            x_str_j += 1
        elif col_type == 'x_nan':
            enc[col] = f'x_nan_{x_nan_k}'
            x_nan_k += 1
        elif col_type == 'x_int':
            enc[col] = f'x_int_{x_int_m}'
            x_int_m += 1
        else:
            enc[col] = f'x_unknown_{col}'

    return enc


def merge_descriptor_target(desc_path, target_path, key='SMILES'):
    desc_df = load_table(desc_path)
    target_df = load_table(target_path)

    if key not in desc_df.columns:
        raise KeyError(f"Key '{key}' not found in descriptor file")
    if key not in target_df.columns:
        raise KeyError(f"Key '{key}' not found in target file")

    desc_smiles = set(desc_df[key].dropna().astype(str))
    target_smiles = set(target_df[key].dropna().astype(str))
    missing_from_desc = sorted(target_smiles - desc_smiles)
    if missing_from_desc:
        print(f"Warning: {len(missing_from_desc)} SMILES entries in target are missing from descriptor file and will be dropped in merged_df:")
        for s in missing_from_desc[:20]:
            print(f"  - {s}")
        if len(missing_from_desc) > 20:
            print(f"  ... and {len(missing_from_desc)-20} more")

    merged = pd.merge(desc_df, target_df, on=key, how='inner')
    encoding = make_descriptor_encoding(desc_df, key=key)

    return merged, encoding

In [ ]:
merged_df, encoding = merge_descriptor_target('2026_03_20_library_pyox_2026_03_03_descriptor_stats.csv', 
                                              'experiment_data_ee.xlsx',
                                              key='INCHI')


In [ ]:
from mlr_utils.feature_selection import auto_feature_selection
from sklearn.linear_model import LinearRegression

# Choose the target column name from target dataset (adjust to your target's actual name)
target_col = 'ΔΔG'  # update as needed for your target column

# Exhaustive search for the best descriptor combinations.
# This will evaluate subset sizes [1,2,3] by default for speed.
result = auto_feature_selection(
    merged_df,
    target_col=target_col,
    key='INCHI',
    selection_mode='sfs',
    direction='forward',
    n_features_to_select=[1, 2],
    split_method='y_equidistant',
    random_state=42,
    scoring='r2',
    model=LinearRegression(),
    max_combinations=20000,
    n_jobs=-1,
    corr_threshold=0.9,  # remove highly correlated features before selection
    exclude_cols=['yield', 'ee', 'Unnamed: 3'],
    print_ml_equation=True,
    top_k=10
)

print(result['mlr_info'])

In [ ]:
top_10 = result['top']
for rank, item in enumerate(top_10, 1):
    print(rank, f"| # of desc:", item['n_features'], "| R2 train :", round(item['train_score'], 2), "| R2 validation :", round(item['val_score'], 2), "| Features :", item['features'])

In [ ]:
from mlr_utils.plot_utils import plot_result_parity

fig, ax, metrics, split_stats = plot_result_parity(
    merged_df,
    target_col=target_col,
    features=result['best']['features'],
    model=LinearRegression(),
    split_method='y_equidistant',
    cv=5,
)

In [ ]:
from mlr_utils.plot_utils import bokeh_parity_with_tooltips
from sklearn.linear_model import LinearRegression

p = bokeh_parity_with_tooltips(
    merged_df,
    target_col='ΔΔG',
    features=result['best']['features'],
    model=LinearRegression(),
    cv=5,
    split_method='y_equidistant',
    inchi_col='INCHI'
)